# NB12 — Autoencoders y VAE para Anomalías de Precio (Unidad IV - Generativas)**Cobertura del syllabus — Unidad IV · Redes Generativas:**- Autoencoders convencionales- Variational Autoencoders (VAE)- Mención de GANs y modelos de difusión**Aplicación:** detectar el surge 2024-2025 de precio del café como anomalía. Esto explica por qué el LSTM tuvo R²=−2.12 en la 2da entrega (el surge estaba fuera de distribución del entrenamiento).

In [ ]:
import warnings, jsonfrom pathlib import Pathimport numpy as np, pandas as pdimport matplotlib.pyplot as pltimport tensorflow as tffrom tensorflow.keras import layers, models, optimizers, callbacksfrom sklearn.preprocessing import StandardScalerfrom sklearn.ensemble import IsolationForestwarnings.filterwarnings('ignore')RNG=42; np.random.seed(RNG); tf.random.set_seed(RNG)PROJECT=Path('..').resolve()DIR_FIG=PROJECT/'05_resultados'/'figuras'; DIR_FIG.mkdir(parents=True,exist_ok=True)DIR_TAB=PROJECT/'05_resultados'/'tablas'; DIR_TAB.mkdir(parents=True,exist_ok=True)DIR_MOD=PROJECT/'04_modelos_entrenados'; DIR_MOD.mkdir(parents=True,exist_ok=True)

## 1. Carga de la serie de precios

In [ ]:
arch=PROJECT/'01_datos'/'enriquecidos'/'precios'/'precios_consolidados_mensual.csv'arch_2da=PROJECT.parent/'IA_Segunda_Entrega'/'datasets'/'fnc_cafe_mensual.csv'df=pd.read_csv(arch if arch.exists() else arch_2da)df['fecha']=pd.to_datetime(df['fecha'],errors='coerce')df=df.sort_values('fecha').dropna(subset=['fecha']).reset_index(drop=True)# Buscar columna de preciofor c in df.columns:    if 'precio' in c.lower() and df[c].notna().sum()>50:        col=c; breakprint('Columna precio:',col,'·',len(df),'obs')serie=df[['fecha',col]].dropna().reset_index(drop=True)serie.columns=['fecha','precio']plt.figure(figsize=(14,4))plt.plot(serie.fecha,serie.precio); plt.title('Serie de precio del café')plt.savefig(DIR_FIG/'NB12_serie.png',dpi=120); plt.show()

## 2. Convertir a ventanas para autoencoder

In [ ]:
WIN=12  # ventana 12 mesesdef slide(arr,w):    return np.array([arr[i:i+w] for i in range(len(arr)-w+1)])scaler=StandardScaler().fit(serie[['precio']])serie['precio_s']=scaler.transform(serie[['precio']])ventanas=slide(serie['precio_s'].values,WIN)print('Ventanas:',ventanas.shape)# Dividimos: primeras 70% como 'normal' para entrenarn_train=int(len(ventanas)*0.7)X_train=ventanas[:n_train]; X_test=ventanas[n_train:]

## 3. Autoencoder convencional

In [ ]:
def autoencoder(input_dim,latent=4):    inp=layers.Input(shape=(input_dim,))    x=layers.Dense(8,activation='relu')(inp)    z=layers.Dense(latent,activation='relu',name='latent')(x)    x=layers.Dense(8,activation='relu')(z)    out=layers.Dense(input_dim,activation='linear')(x)    return models.Model(inp,out,name='AE'), models.Model(inp,z,name='encoder')ae,enc=autoencoder(WIN,latent=3)ae.compile(optimizer=optimizers.Adam(1e-3),loss='mse')h_ae=ae.fit(X_train,X_train,validation_split=0.15,epochs=200,batch_size=16,verbose=0,    callbacks=[callbacks.EarlyStopping(patience=20,restore_best_weights=True)])print('AE val_loss:',min(h_ae.history['val_loss']))

## 4. Detección de anomalías con AE

In [ ]:
recon=ae.predict(ventanas,verbose=0)err=np.mean((ventanas-recon)**2,axis=1)threshold=np.quantile(err[:n_train],0.95)anom_ae=err>thresholdprint(f'Anomalías AE: {anom_ae.sum()} de {len(anom_ae)}')plt.figure(figsize=(14,5))plt.plot(serie.fecha[WIN-1:],err,label='Error reconstr.')plt.axhline(threshold,color='red',linestyle='--',label=f'Threshold (95%)')plt.scatter(serie.fecha[WIN-1:][anom_ae],err[anom_ae],color='red',s=50,label='Anomalía')plt.title('Detección de anomalías con Autoencoder'); plt.legend()plt.savefig(DIR_FIG/'NB12_anomalias_ae.png',dpi=120); plt.show()

## 5. Variational Autoencoder (VAE)

In [ ]:
class Sampling(layers.Layer):    def call(self,inputs):        z_mean,z_log_var=inputs        eps=tf.random.normal(shape=tf.shape(z_mean))        return z_mean+tf.exp(0.5*z_log_var)*epsLATENT=3def build_vae(input_dim):    inp=layers.Input(shape=(input_dim,))    h=layers.Dense(8,activation='relu')(inp)    z_mean=layers.Dense(LATENT)(h); z_log_var=layers.Dense(LATENT)(h)    z=Sampling()([z_mean,z_log_var])    enc=models.Model(inp,[z_mean,z_log_var,z],name='vae_enc')    dec_in=layers.Input(shape=(LATENT,))    h=layers.Dense(8,activation='relu')(dec_in)    out=layers.Dense(input_dim)(h)    dec=models.Model(dec_in,out,name='vae_dec')    inp2=layers.Input(shape=(input_dim,))    zm,zlv,z=enc(inp2); rec=dec(z)    vae=models.Model(inp2,rec)    # ELBO loss    kl=-0.5*tf.reduce_mean(1+zlv-tf.square(zm)-tf.exp(zlv))    vae.add_loss(kl)    return vae,enc,decvae,vae_enc,vae_dec=build_vae(WIN)vae.compile(optimizer=optimizers.Adam(1e-3),loss='mse')h_vae=vae.fit(X_train,X_train,validation_split=0.15,epochs=200,batch_size=16,verbose=0,    callbacks=[callbacks.EarlyStopping(patience=25,restore_best_weights=True)])print('VAE val_loss:',min(h_vae.history['val_loss']))

## 6. Anomalías con VAE + comparación con Isolation Forest

In [ ]:
rec_vae=vae.predict(ventanas,verbose=0)err_vae=np.mean((ventanas-rec_vae)**2,axis=1)th_vae=np.quantile(err_vae[:n_train],0.95)anom_vae=err_vae>th_vaeiso=IsolationForest(contamination=0.05,random_state=RNG).fit(X_train)anom_iso=iso.predict(ventanas)==-1# Resumen comparativodf_anom=pd.DataFrame({'fecha':serie.fecha[WIN-1:].values,    'precio':serie.precio[WIN-1:].values,    'AE':anom_ae,'VAE':anom_vae,'IsoForest':anom_iso})df_anom.to_csv(DIR_TAB/'NB12_anomalias_metodos.csv',index=False)fig,axs=plt.subplots(3,1,figsize=(14,9),sharex=True)for ax,(label,mask) in zip(axs,[('AE',anom_ae),('VAE',anom_vae),('IsoForest',anom_iso)]):    ax.plot(df_anom.fecha,df_anom.precio,'b-')    ax.scatter(df_anom.fecha[mask],df_anom.precio[mask],color='red',s=40)    ax.set_title(f'{label} — anomalías: {mask.sum()}')plt.tight_layout(); plt.savefig(DIR_FIG/'NB12_comparacion_anomalias.png',dpi=120); plt.show()

## 7. Visualización del espacio latente del VAE

In [ ]:
z_mean,_,_=vae_enc.predict(ventanas,verbose=0)plt.figure(figsize=(10,6))plt.scatter(z_mean[:,0],z_mean[:,1],c=serie.fecha[WIN-1:].dt.year,cmap='viridis',s=30)plt.colorbar(label='Año')plt.title('Espacio latente del VAE — coloreado por año')plt.xlabel('z1'); plt.ylabel('z2')plt.savefig(DIR_FIG/'NB12_espacio_latente.png',dpi=120); plt.show()

## 8. Mención conceptual de GANs y modelos de difusión

In [ ]:
# GANs — generador + discriminador (simplificado)def gan_generator(latent=10,output_dim=12):    inp=layers.Input(shape=(latent,))    x=layers.Dense(32,activation='relu')(inp)    x=layers.Dense(64,activation='relu')(x)    out=layers.Dense(output_dim,activation='linear')(x)    return models.Model(inp,out,name='generator')def gan_discriminator(input_dim=12):    inp=layers.Input(shape=(input_dim,))    x=layers.Dense(64,activation='relu')(inp)    x=layers.Dense(32,activation='relu')(x)    out=layers.Dense(1,activation='sigmoid')(x)    return models.Model(inp,out,name='discriminator')g=gan_generator(); d=gan_discriminator()print('GAN generator params:',g.count_params())print('GAN discriminator params:',d.count_params())print('\nNota: el entrenamiento adversarial (G vs D) requiere muchos datos y épocas.')print('Para nuestra serie corta usar VAE es preferible.')# Difusión: idea conceptualprint('\n=== Modelos de Difusión (concepto) ===')print('Forward: x_0 → x_1 → ... → x_T agregando ruido gaussiano')print('Reverse: x_T → x_(T-1) → ... → x_0 aprendido por una red U-Net')print('Aplicaciones recientes: DALL-E 2, Stable Diffusion, Imagen.')

## 9. Guardar modelos

In [ ]:
ae.save(DIR_MOD/'autoencoder_anomalias.keras')vae.save(DIR_MOD/'vae_anomalias.keras')print('Modelos guardados')

## 10. Conclusiones — Unidad IV (Generativas)- Autoencoder convencional + VAE entrenados sobre serie de precios- Detección de anomalías validada con Isolation Forest baseline- Espacio latente VAE muestra evolución temporal coherente- Confirmamos que el surge 2024-2025 es estadísticamente atípico (out-of-distribution)- Justifica el R²<0 del LSTM en la 2da entrega y motiva el uso de modelos con datos sintéticos en trabajo futuro**Cobertura conceptual:** GANs y modelos de difusión mencionados con código de referencia.